# How far can time series take us?
## S&P 500 forecasting with an honest ARIMA baseline

**Runnable AI-assisted academic reconstruction · 14 October 2026 · Python**

This notebook reconstructs an analysis from a short supplied project note. It does **not** recover the original college dataset, code, forecast protocol, or results. The historical evaluation is simulated retrospectively using the downloaded vintage. AI assisted in building this reconstruction; the numerical predictions come from a fitted statistical model.

The study asks: **Does ARIMA(1,1,1) improve on the last observed S&P 500 close, and how does the answer change with the forecast horizon?**

## tl;dr
The following summary is calculated from this execution, so the numbers update if the source vintage changes. A model line that visually follows actual prices can still add very little over yesterday's close.

In [ ]:
from pathlib import Path
from contextlib import redirect_stdout
import io
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import analysis
get_ipython().run_line_magic('matplotlib', 'inline')

ROOT = Path.cwd()
assert (ROOT / 'analysis.py').exists(), 'Run this notebook from the sp500_project folder.'
with redirect_stdout(io.StringIO()):
    summary = analysis.run(output_root=ROOT)
metrics = pd.DataFrame(summary['metrics'])
rolling_one = metrics.query("protocol == 'Rolling origin' and horizon_sessions == '1'").set_index('model')
arima_one = rolling_one.loc['ARIMA(1,1,1)']
naive_one = rolling_one.loc['Naive']
fixed_metrics = metrics.query("protocol == 'Fixed origin'").set_index('model')
stress_metrics = metrics.query("protocol == 'Rolling origin 2020'").set_index('model')
display(Markdown(f'''- **Next-session accuracy:** ARIMA RMSE **{arima_one.rmse_points:.2f} points**, versus naive **{naive_one.rmse_points:.2f}**: only **{arima_one.rmse_improvement_vs_naive_pct:.2f}%** lower RMSE in 2014–2020.
- **Long-horizon limitation:** fixed-origin ARIMA RMSE is **{fixed_metrics.loc['ARIMA(1,1,1)', 'rmse_points']:.2f} points**; its no-drift mean does not reproduce the sustained increase in the test data.
- **Uncertainty limitation:** nominal 95% one-session intervals cover **{arima_one.arima_95pct_interval_coverage_pct:.1f}%** of actual closes, falling to **{stress_metrics.loc['ARIMA(1,1,1)', 'arima_95pct_interval_coverage_pct']:.1f}%** in 2020. The constant-variance model is poorly calibrated in that slice.
- No statistical superiority, trading profitability, or improved performance from untested AI/external-factor extensions is claimed.'''))

## Context & Methods

The [supplied project note](https://www.linkedin.com/feed/update/urn:li:activity:7443069084801454080/) mentions historical S&P 500 data, ARIMA(1,1,1), ADF, MLE, and RMSE/MAE. It does not identify its exact source file, frequency, drift specification, refitting policy, or baseline results. Here those choices are fixed and reproducible.

### Key assumptions

| Choice | This reconstruction |
|---|---|
| Target | Daily `^GSPC` Close, in S&P 500 index points; excludes dividend reinvestment |
| Development | Observations through 31 December 2013 |
| Historical test | 2 January 2014–31 December 2020 |
| ARIMA | Prespecified (1,1,1), no drift, Gaussian state-space MLE |
| Model tuning | No claim of tuning; order inherited from the note |
| Rolling updates | Update filtered state only, after forecasting; coefficients and variance frozen at end-2013 |
| Horizons | 1, 5, and 20 observed trading sessions |
| Baselines | Last close; last close plus training-average daily change × horizon |
| Evaluation | MAE/RMSE on identical target rows within each protocol and horizon |
| Data vintage | Downloaded 14 October 2026; not verified contemporaneous point-in-time data |

There is no shuffled split, target leakage through preprocessing, or interpolation across weekends and holidays. Future known index closes are used only to score forecasts or, after their simulated availability, update the rolling state. No test-period observations are used to estimate parameters.

### Two questions, two information sets

**Fixed origin:** stand at 31 December 2013 and forecast every future test close without seeing later prices. This pools horizons 1 through 1,763 sessions in its overall error.

**Rolling origin:** at each session, forecast ahead with the closes already observed. Afterward, reveal the next actual close and update the filter. A one-session forecast therefore normally starts close to yesterday's market level. These two error summaries measure different tasks and should not be ranked against each other as if the horizons were identical.

## Data
### 1. Read and validate the source
The supplied CSV uses quoted comma-decimal values; the loader explicitly normalizes them to numeric values. No row is silently discarded. The hash verifies the local file against its metadata, while the calendar comparison checks coverage against XNYS sessions. Neither check validates the exact close values against another provider.

In [ ]:
frame = analysis.load_data(ROOT / 'local_data' / 'yahoo_gspc_2000_2020_close.csv')
audit = summary['data_audit']
display(pd.DataFrame([{'Rows': audit['observations'], 'First date': audit['first_date'],
                      'Last date': audit['last_date'], 'Missing closes': audit['nulls'],
                      'Duplicates': audit['duplicates'], 'Missing XNYS sessions': len(audit['missing_expected_sessions']),
                      'Interpolated rows': audit['interpolated_rows']}]))
display(frame.iloc[[0, 1, -2, -1]].round(2))

### 2. Explore levels and daily returns
This full-period chart is a descriptive overview. The parameter choice and statistical diagnostics below use the development period only. Returns here describe relative changes; the forecasting target remains the index level.

In [ ]:
analysis.configure_charts()
fig, axes = plt.subplots(2, 1, figsize=(11, 6.8), sharex=True)
axes[0].plot(frame.index, frame.Close, color=analysis.COLORS['Actual'], lw=1)
axes[0].axvspan(pd.Timestamp('2014-01-01'), frame.index[-1], color='#E6F0F5', label='Historical test')
axes[0].set(title='S&P 500 observed daily close, 2000–2020', ylabel='Index points')
axes[0].legend(frameon=False)
axes[1].plot(frame.index, frame.Close.pct_change() * 100, color=analysis.COLORS['ARIMA(1,1,1)'], lw=.6)
axes[1].set(title='Daily percentage changes reveal varying volatility', ylabel='Daily return (%)', xlabel='Date')
for ax in axes: analysis.date_axis(ax, frame.index)
fig.tight_layout()
display(fig)
plt.close(fig)

Large positive and negative daily moves cluster together. A trend in levels does not mean that tomorrow's change is easy to forecast. Point-level errors also depend on the changing index scale, so later years are not directly comparable to early years as risk-adjusted performance.

## Results
### 3. Check the unit-root hypothesis on training observations
The ADF **null hypothesis is a unit root**. The regression includes a constant, with lag length selected by AIC. A large p-value is a failure to reject, not proof that a unit root exists. A small p-value after differencing supports using that transformed series under this specification; it does not prove constant variance.

In [ ]:
adf_table = pd.DataFrame(summary['diagnostics']['adf'])
display(adf_table[['series', 'statistic', 'p_value', 'lags', 'nobs', 'reject_at_5pct']])
train = frame.loc[:analysis.TRAIN_END, 'Close'].to_numpy()
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
plot_acf(np.diff(train), lags=30, zero=False, ax=axes[0], title='Training first differences: ACF')
plot_pacf(np.diff(train), lags=30, zero=False, method='ywm', ax=axes[1], title='Training first differences: PACF')
for ax in axes: ax.set(xlabel='Lag (trading sessions)', ylabel='Correlation')
fig.tight_layout()
display(fig)
plt.close(fig)

The training level fails to reject the unit-root null; its first difference rejects it. The low-order ARIMA is a transparent starting specification inherited from the note. These plots do not establish that (1,1,1) is the optimal order, and no test-period search is performed.

### 4. Understand the model and fit
For close $y_t$, let $\Delta y_t=y_t-y_{t-1}$. The fitted no-drift model is

$$\Delta y_t = \phi\Delta y_{t-1} + \varepsilon_t + \theta\varepsilon_{t-1}.$$

`p=1` uses one lagged difference, `d=1` takes one difference, and `q=1` uses one previous innovation. The MA term is an innovation/error term; it is not a moving average of raw prices. MLE chooses the coefficients and innovation variance using the training likelihood. With no drift and a stable differenced process, long-horizon expected changes decay toward zero.

The implementation in `analysis.py` explicitly calls `ARIMA(..., order=(1,1,1), trend='n')`, then checks optimizer convergence. AIC/BIC below describe this fit; they are not evidence of a model search.

In [ ]:
diag = summary['diagnostics']
display(pd.DataFrame(diag['parameters'].items(), columns=['Parameter', 'Training estimate']))
display(pd.DataFrame([{'Converged': diag['optimizer_converged'], 'Iterations': diag['optimizer_iterations'],
                      'AIC': diag['aic'], 'BIC': diag['bic'],
                      'Drift baseline (points/session)': summary['model']['drift_baseline_points_per_session']}]))

### 5. Inspect residuals before trusting the fit
The diffuse initialization residual is excluded. Ljung–Box checks residual autocorrelation, allowing for two fitted AR/MA coefficients. ARCH LM tests for changing conditional variance at ten lags. Passing one selected autocorrelation check does not validate every aspect of a model.

In [ ]:
display(pd.DataFrame(diag['ljung_box']))
display(pd.DataFrame([diag['arch_lm']]))
from IPython.display import Image
display(Image(filename=str(ROOT / 'charts' / '07_residual_diagnostics.png')))

At lag 10 the Ljung–Box test does not reject at 5%; at lag 20 it rejects. The very small ARCH LM p-value and dependence in squared residuals are evidence against constant conditional variance. The model is useful as a baseline but does not pass a blanket residual-adequacy claim.

### 6. Fixed-origin historical forecast
All future test points below are forecast from the same historical information set. The shaded band is the ARIMA nominal 95% prediction interval, conditional on estimated coefficients and constant innovation variance; it excludes parameter uncertainty.

In [ ]:
fixed = pd.read_csv(ROOT / 'results' / 'fixed_origin_forecasts.csv', parse_dates=['Date', 'Origin'])
rolling = pd.read_csv(ROOT / 'results' / 'rolling_forecasts.csv', parse_dates=['Date', 'Origin'])
fig, ax = plt.subplots(figsize=(11, 5.2))
ax.fill_between(fixed.Date, fixed.Lower95, fixed.Upper95, color='#126B98', alpha=.12, label='ARIMA nominal 95% interval')
for name, style in [('Actual', '-'), ('ARIMA(1,1,1)', '-'), ('Naive', '--'), ('Drift', ':')]:
    ax.plot(fixed.Date, fixed[name], label=name, color=analysis.COLORS[name], ls=style, lw=1.5)
ax.set(title='Forecast origin: 31 December 2013; no later closes supplied', ylabel='Index points', xlabel='Target date')
analysis.date_axis(ax, fixed.Date)
ax.legend(frameon=False, fontsize=10)
fig.tight_layout()
display(fig)
plt.close(fig)
display(fixed_metrics[['n', 'mae_points', 'rmse_points']].round(2))

The no-drift ARIMA mean stays close to the final training level and does not recover the sustained rise in actual closes. The frozen positive drift baseline does better in this specific test period but still substantially underestimates the later index level. This does not prove that drift will be better in future regimes.

### 7. Rolling-origin forecast: tracking versus predictive value
At every origin, the algorithm forecasts first and only then reveals the next close to `extend()`. The coefficient vector is asserted unchanged on every update. The 2020 zoom makes the reaction to new prices visible; ARIMA and the last-close baseline are often very similar.

In [ ]:
one = rolling.loc[rolling.Horizon.eq(1)].copy()
zoom = one.loc[one.Date.between('2020-02-18', '2020-04-30')]
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
for ax, data, title in [(axes[0], one, 'Rolling one-session forecasts, 2014–2020'),
                         (axes[1], zoom, '2020 close-up: reacting to prior-session information')]:
    for name, style in [('Actual', '-'), ('ARIMA(1,1,1)', '--'), ('Naive', ':')]:
        ax.plot(data.Date, data[name], label=name, color=analysis.COLORS[name], ls=style, lw=1.5)
    ax.set(title=title, ylabel='Index points', xlabel='Target date')
    analysis.date_axis(ax, data.Date)
    ax.legend(frameon=False, ncol=3, fontsize=10, loc='upper right')
fig.tight_layout()
display(fig)
plt.close(fig)
display(rolling_one[['n', 'mae_points', 'rmse_points', 'rmse_improvement_vs_naive_pct']].round(3))

### 8. Compare identical forecast horizons
For $n$ forecasts, $\mathrm{MAE}=\frac{1}{n}\sum|y_t-\hat y_t|$ and $\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum(y_t-\hat y_t)^2}$. RMSE penalizes large errors more strongly. Both are in index points.

Improvement over naive is $100\times(1-\mathrm{error}_{model}/\mathrm{error}_{naive})$; a negative number means the model is worse. There are fewer eligible targets at longer horizons because every forecast must resolve by 31 December 2020. Comparisons within each horizon use exactly the same targets.

In [ ]:
horizon_metrics = metrics.query("protocol == 'Rolling origin'")
display(horizon_metrics[['horizon_sessions', 'model', 'n', 'mae_points', 'rmse_points', 'rmse_improvement_vs_naive_pct']].round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
for ax, measure, title in [(axes[0], 'mae_points', 'MAE'), (axes[1], 'rmse_points', 'RMSE')]:
    positions = np.arange(3)
    for offset, name in zip([-.25, 0, .25], ['Naive', 'Drift', 'ARIMA(1,1,1)']):
        selected = horizon_metrics.loc[horizon_metrics.model.eq(name)].set_index('horizon_sessions').loc[['1','5','20']]
        bars = ax.bar(positions + offset, selected[measure], width=.24, label=name, color=analysis.COLORS[name])
        ax.bar_label(bars, fmt='%.1f', fontsize=9, padding=2)
    ax.set(title=title, ylabel='Index points', xlabel='Horizon (trading sessions)')
    ax.set_xticks(positions, ['1', '5', '20'])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
    ax.grid(axis='y'); ax.set_axisbelow(True)
axes[0].legend(frameon=False, fontsize=10)
fig.tight_layout()
display(fig)
plt.close(fig)

ARIMA's one-session RMSE advantage is small and does not persist at five or twenty sessions in this experiment. The longer-horizon errors overlap across origins, so these descriptive differences are not evidence of statistical significance. No transaction-cost or trading-rule experiment was performed.

### 9. Check interval coverage and the 2020 slice
Coverage is the fraction of actual closes inside each model-based 95% prediction interval. The nominal label is an assumption-driven target; it is not a guarantee. The 2020 slice is a descriptive examination of the same holdout, not a new independent holdout selected after the fact.

In [ ]:
coverage = metrics.loc[metrics.model.eq('ARIMA(1,1,1)'), ['protocol', 'horizon_sessions', 'n', 'arima_95pct_interval_coverage_pct']]
display(coverage.round(2))
annual = pd.read_csv(ROOT / 'results' / 'annual_one_step.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].plot(annual.Year, annual.ARIMA_RMSE, 'o-', label='ARIMA', color='#126B98')
axes[0].plot(annual.Year, annual.Naive_RMSE, 'x--', label='Naive', color='#A06C22')
axes[0].set(title='One-session RMSE by year', xlabel='Year', ylabel='Index points', ylim=(0, None))
axes[0].legend(frameon=False)
axes[1].bar(annual.Year, annual.Coverage_pct, color='#126B98')
axes[1].axhline(95, ls='--', color='#253247', label='Nominal 95%')
axes[1].set(title='ARIMA one-session interval coverage', xlabel='Year', ylabel='Covered actual closes (%)', ylim=(0, 105))
axes[1].legend(frameon=False)
for ax in axes: ax.grid(axis='y'); ax.set_axisbelow(True); ax.set_xticks(annual.Year)
fig.tight_layout()
display(fig)
plt.close(fig)
display(stress_metrics[['n', 'mae_points', 'rmse_points']].round(2))

### 10. Verification and reproducibility
The script checks chronological origins, forecast counts, frozen coefficients, convergence, calendar coverage, metadata hash, and every MAE/RMSE via a second scalar calculation. The check below additionally rebuilds the naive one-session forecast directly from shifted observed closes, independent of the forecasting loop.

In [ ]:
expected_naive = frame.Close.shift(1).reindex(pd.DatetimeIndex(one.Date)).to_numpy()
np.testing.assert_allclose(one.Naive, expected_naive, rtol=0, atol=1e-10)
assert (rolling.Origin < rolling.Date).all()
assert one.Date.iloc[0] == pd.Timestamp('2014-01-02')
assert one.Date.iloc[-1] == pd.Timestamp('2020-12-31')
assert len(one) == 1763
print('Independent shifted-close benchmark check: passed')
print('Chronological target/origin checks: passed')
display(pd.DataFrame(summary['validation']['packages'].items(), columns=['Package', 'Executed version']))

## Takeaways

1. **The forecast protocol is central to the result.** A rolling line that tracks the market uses fresh observations. It is not a seven-year forecast made in 2013.
2. **Start with the naive baseline.** The reconstructed ARIMA offers only a small one-session error reduction and no advantage at the two longer rolling horizons tested.
3. **Model the uncertainty, not only the mean.** Residual dependence, changing volatility, and poor interval coverage expose important limitations of this constant-variance baseline.
4. **Expand with an explicit experiment.** A refit schedule, log-price/return target, ARIMAX predictors, or a volatility model should be specified and validated before evaluation. More complexity and AI do not guarantee improvement.

### What the supplied note should change

Avoid claiming that ARIMA 'captures the overall market trend' without naming its information set and benchmark. Also distinguish ARIMAX from GARCH: ARIMAX adds external predictors; GARCH models conditional variance. Neither was tested here, and neither can be presented as a demonstrated improvement.

### AI assistance and learner ownership

AI supported scoping, code generation, drafting, and verification. There is no LLM numerical forecaster in this experiment. The learner should be able to explain the split, origin, baselines, error formulas, residual findings, and limits, and should disclose assistance under the applicable course policy. The underlying original analysis and the course's specific AI policy were not supplied to this notebook.

### Boundaries

This is retrospective historical analysis, not a 2026 market forecast or trading recommendation. Original project identity is unverified. Cross-provider close verification and contemporaneous vintage verification were not performed. The source metadata does not establish an open redistribution licence. Keep local raw data and detailed actual-price extracts private pending a rights review.

### Sources and implementation

- [Yahoo Finance ^GSPC history](https://finance.yahoo.com/quote/%5EGSPC/history/); exact endpoint and hash are in `local_data/source_metadata.json`.
- [Statsmodels ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html), [state-space forecasting](https://www.statsmodels.org/stable/examples/notebooks/generated/statespace_forecasting.html), and [ADF](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html).
- [exchange_calendars](https://github.com/gerrymanoim/exchange_calendars) provides the XNYS calendar proxy.
- `analysis.py` holds the complete computational functions; `requirements.txt` pins the primary numerical packages; `results/summary.json` retains exact results and run semantics.

The stable web documentation can change; the package versions in the executed output identify this run.